# M2 QA: existing parse output

QA has a separate run ID, revision, timing and evidence. It does not rerun parsing or change the M2 benchmark. This notebook is being verified; do not treat its output as official until the full validation is complete.


In [ ]:
from pathlib import Path

M2_DRIVE_DIR = "/content/drive/MyDrive/arxiv-nhom1/m2"
M2_RUN_ID = "m2-full-01"
QA_RUN_ID = "qa-full-01"
LOCAL_QA_ROOT = "/content/m2-qa"
CODE_REF = "main"
REPO = "https://github.com/hcmus-arxiv-nhom1/CodeRepo.git"
RUN_QA = False  # Set True only when ready to run the checks.
MAX_SHARD_ROWS = 500_000
EXPECTED_SHARDS = 200  # Known inventory for m2-full-01; adjust for another run.

M1_DRIVE_DIR = "/content/drive/MyDrive/arxiv-nhom1"
LOCAL_CORPUS_ROOT = "/content/m2-local"
REUSE_PARTIAL_RESTORE = False  # True only when no other process writes these folders.
RESTORE_WORKERS = 8
RUN_SOURCE_QA = False  # Optional: restore M1 and run source-based audits.
RUN_SCORE = False  # Set True only after filling the audit sheet.

## Mount Drive and select the QA revision

A new QA run resolves `main` to a commit; an existing QA ID uses its saved revision. Use a CPU runtime and do not run two sessions with the same QA ID.


In [ ]:
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import time

from google.colab import drive

drive.mount("/content/drive")
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,127}", QA_RUN_ID):
    raise ValueError("Invalid QA run ID")
SOURCE = Path(M2_DRIVE_DIR) / M2_RUN_ID
QA_DESTINATION = SOURCE / "qa" / QA_RUN_ID
LOCAL = Path(LOCAL_QA_ROOT) / QA_RUN_ID
revision = CODE_REF
if (QA_DESTINATION / "identity.json").is_file():
    revision = json.loads((QA_DESTINATION / "identity.json").read_text())["qa_revision"]

In [ ]:
if any(name == "src" or name.startswith("src.") for name in sys.modules):
    raise RuntimeError("Use a fresh Python kernel before checking out a QA revision.")
repo_dir = Path("/content/CodeRepo-qa")
if not (repo_dir / ".git").is_dir():
    subprocess.run(["git", "clone", "--no-checkout", REPO, str(repo_dir)], check=True)
subprocess.run(["git", "-C", str(repo_dir), "fetch", "origin", revision], check=True)
subprocess.run(["git", "-C", str(repo_dir), "checkout", "--detach", "FETCH_HEAD"], check=True)
os.chdir(repo_dir)
if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))
QA_REVISION = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("QA revision:", QA_REVISION)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "pyarrow", "pandas", "tqdm", "psutil"],
    check=True,
)

## Verify run identity

The identity includes parser and QA revisions, source manifest, journal and shard-hash manifest digests. The benchmark files are read-only inputs.


In [ ]:
from src.qa.qa_inputs import inspect_run, prepare_qa

if RUN_QA:
    qa_table_passed = False
    qa_structure_complete = False
    qa_sources_verified = False
    identity, hashes = prepare_qa(SOURCE, M2_RUN_ID, QA_RUN_ID, QA_REVISION, EXPECTED_SHARDS)
    print("Parser revision:", identity["parser_revision"])
    print("Committed shards:", len(hashes))
else:
    print("QA disabled; set RUN_QA = True to prepare a QA run.")

## Restore the committed outputs

Copies are verified against the recorded shard hashes before publication. Existing verified local shards are reused; unexpected or changed local files stop the run.


In [ ]:
from tqdm.auto import tqdm

from src.qa.qa_inputs import restore_shards

if RUN_QA:
    qa_table_passed = False
    qa_structure_complete = False
    qa_sources_verified = False
    LOCAL.mkdir(parents=True, exist_ok=True)
    local_out = LOCAL / "out"
    with tqdm(total=len(hashes), desc="Restore QA inputs", unit="shard") as bar:

        def restored(done, total):
            bar.update(done - bar.n)

        print(restore_shards(SOURCE / "out", local_out, hashes, on_progress=restored))
    current, _ = inspect_run(SOURCE, M2_RUN_ID, QA_REVISION, EXPECTED_SHARDS)
    if any(current[key] != value for key, value in identity.items() if key != "qa_id"):
        raise ValueError("Source run changed during restore")
    journal_bytes = (SOURCE / "work/shards.jsonl").read_bytes()
    if hashlib.sha256(journal_bytes).hexdigest() != identity["journal_sha256"]:
        raise ValueError("Source journal changed")
    local_journal = LOCAL / "shards.jsonl"
    if local_journal.exists() and local_journal.read_bytes() != journal_bytes:
        raise ValueError("Existing local journal differs")
    if not local_journal.exists():
        temp_journal = LOCAL / ".shards.part"
        temp_journal.write_bytes(journal_bytes)
        os.replace(temp_journal, local_journal)

## Table integrity

Validation and graph checks report shard progress separately. QA checkpoints are stored under the QA run ID and bound to input hashes and QA revision. The row cap refuses a shard that is too large instead of silently omitting it.


In [ ]:
from src.qa.check_tables_bounded import check_run

if RUN_QA:
    qa_table_passed = False
    qa_structure_complete = False
    qa_started = time.perf_counter()
    with (
        tqdm(total=len(hashes), desc="Validate partition", unit="shard") as validation,
        tqdm(total=len(hashes), desc="Check tables", unit="shard") as checking,
    ):

        def update_validation(done, total):
            validation.update(done - validation.n)

        def update_check(done, total):
            checking.update(done - checking.n)

        table_report = check_run(
            local_out,
            local_journal,
            max_shard_rows=MAX_SHARD_ROWS,
            checkpoint_dir=QA_DESTINATION / "checkpoints",
            identity=identity,
            on_validation=update_validation,
            expected_hashes=hashes,
            on_progress=update_check,
        )
    table_report["qa_session_wall_seconds"] = time.perf_counter() - qa_started
    table_report["timing_scope"] = "current_session_only"
    report_path = QA_DESTINATION / "check_tables.json"
    temp_report = QA_DESTINATION / ".check_tables.part"
    temp_report.write_text(json.dumps(table_report, indent=2) + "\n", encoding="utf8")
    os.replace(temp_report, report_path)
    print(json.dumps(table_report, indent=2))
    if not table_report["ok"]:
        raise RuntimeError("Table integrity failed; inspect check_tables.json")
    qa_table_passed = True

## Restore and verify the original corpus

Structure comparison needs the M1 sources. The original pinned checkpoint indexes are verified and existing local files can be reused. This can take time and is outside the parser benchmark.

Optional: set `RUN_SOURCE_QA = True` to run this section; it requires restoring the M1 corpus.


In [ ]:
from src.benchmark.m2_restore import restore_m2

if RUN_QA and RUN_SOURCE_QA:
    qa_sources_verified = False
    qa_structure_complete = False
    manifest_bytes = (SOURCE / "M2_MANIFEST.json").read_bytes()
    if hashlib.sha256(manifest_bytes).hexdigest() != identity["manifest_sha256"]:
        raise ValueError("M2 manifest changed")
    parse_manifest = json.loads(manifest_bytes)
    DATA_ROOTS = []
    for run_id, rel in parse_manifest["runs"].items():
        remote = Path(M1_DRIVE_DIR) / rel
        index_bytes = (remote / "INDEX.json").read_bytes()
        expected = parse_manifest["crawl_index_sha256"][run_id]
        if hashlib.sha256(index_bytes).hexdigest() != expected:
            raise ValueError(f"M1 index changed: {run_id}")
        index = json.loads(index_bytes)
        count = sum(
            name.startswith("data/corpus/") or name == "work/versions.jsonl"
            for name in index["files"]
        )
        with tqdm(total=count, desc=f"Restore source {run_id}", unit="file") as bar:

            def update_source(event):
                bar.update(event["completed"] - bar.n)

            restore_m2(
                Path(LOCAL_CORPUS_ROOT) / run_id,
                remote,
                index,
                workers=RESTORE_WORKERS,
                reuse_partial=REUSE_PARTIAL_RESTORE,
                expected_index_sha256=expected,
                on_progress=update_source,
            )
        DATA_ROOTS.append(Path(LOCAL_CORPUS_ROOT) / run_id / "data/corpus")
    qa_sources_verified = True

## Structural comparison

Compares source counts to the parsed tree one shard at a time. Checkpoints are separate from the parsing run.

Optional: set `RUN_SOURCE_QA = True` to run this section; it requires restoring the M1 corpus.


In [ ]:
from src.qa.audit_structure import audit_resumable

if RUN_QA and RUN_SOURCE_QA:
    qa_structure_complete = False
    if not globals().get("qa_table_passed", False):
        raise RuntimeError("Complete table integrity checks before structural audit")
    if not globals().get("qa_sources_verified", False):
        raise RuntimeError("Restore and verify all source corpora before structural audit")
    qa_table_passed = False
    with tqdm(total=len(hashes), desc="Compare structures", unit="shard") as bar:

        def structure_progress(done, total):
            bar.update(done - bar.n)

        structure_summary = audit_resumable(
            DATA_ROOTS,
            local_out,
            QA_DESTINATION,
            identity,
            on_progress=structure_progress,
            max_shard_rows=MAX_SHARD_ROWS,
            journal=local_journal,
            expected_hashes=hashes,
        )
    qa_table_passed = True
    qa_structure_complete = True
    print("Structural comparison:", structure_summary)

## Manual audit sample

Draw 30 papers with seed 2026. A team member must fill in the review sheet; an empty sheet is not a successful human audit. Existing sheets are preserved.

Optional: set `RUN_SOURCE_QA = True` to run this section; it requires restoring the M1 corpus.


In [ ]:
from src.qa.audit_sample import draw, eligible_papers
from src.qa.qa_inputs import verify_checkpoint

if RUN_QA and RUN_SOURCE_QA:
    import hashlib
    import json
    import tempfile

    if not globals().get("qa_table_passed", False) or not globals().get(
        "qa_sources_verified", False
    ):
        raise RuntimeError("Complete table checks and verify sources before drawing the sample")
    previous_structure_complete = globals().get("qa_structure_complete", False)
    qa_table_passed = False
    qa_structure_complete = False
    if hashlib.sha256(local_journal.read_bytes()).hexdigest() != identity["journal_sha256"]:
        raise ValueError("Local journal changed before sample")
    with tqdm(total=len(hashes), desc="Verify sample inputs", unit="shard") as bar:

        def sample_verify(done, total):
            bar.update(done - bar.n)

        verify_checkpoint(local_out, local_journal, hashes, on_progress=sample_verify)
    sample_report = QA_DESTINATION / "audit30"
    sample_identity = sample_report / "sample_identity.json"
    if (sample_report / "audit_sheet.csv").exists():
        if (
            not sample_identity.is_file()
            or json.loads(sample_identity.read_text(encoding="utf8")) != identity
        ):
            raise ValueError(
                "Existing sample identity is missing or different; it was not modified"
            )
        print("Existing audit sheet preserved:", sample_report / "audit_sheet.csv")
    elif sample_report.exists():
        raise FileExistsError("Partial audit sample exists; inspect it before retrying.")
    else:
        total = min(30, len(eligible_papers(LOCAL)))
        with tempfile.TemporaryDirectory(prefix=".qa-sample-", dir=QA_DESTINATION) as temp:
            staged = Path(temp) / "audit30"
            with tqdm(total=total, desc="Draw sample", unit="paper") as bar:

                def sample_progress(done, count):
                    bar.update(done - bar.n)

                sheet = draw(
                    local_out,
                    LOCAL,
                    staged,
                    n=30,
                    seed=2026,
                    data_root=DATA_ROOTS,
                    on_progress=sample_progress,
                )
            with tqdm(total=len(hashes), desc="Reverify sample inputs", unit="shard") as bar:

                def sample_reverify(done, count):
                    bar.update(done - bar.n)

                verify_checkpoint(local_out, local_journal, hashes, on_progress=sample_reverify)
            if hashlib.sha256(local_journal.read_bytes()).hexdigest() != identity["journal_sha256"]:
                raise ValueError("Local journal changed during sample")
            (staged / "sample_identity.json").write_text(
                json.dumps(identity, sort_keys=True), encoding="utf8"
            )
            if sample_report.exists():
                raise FileExistsError(
                    "Audit sample appeared while drawing; it was not overwritten."
                )
            staged.rename(sample_report)
        print("Manual review sheets created:", len(sheet))
    qa_table_passed = True
    qa_structure_complete = previous_structure_complete

## Score the human review

Fill the sheet first, then set `RUN_SCORE = True` and run this cell. Blank and NA entries are not treated as reviewed.

Optional: set `RUN_SOURCE_QA = True` to run this section; it requires restoring the M1 corpus.


In [ ]:
from src.qa.audit_sample import score

if RUN_QA and RUN_SOURCE_QA and RUN_SCORE:
    qa_score_generated = False
    import csv
    import hashlib

    audit_sheet = QA_DESTINATION / "audit30/audit_sheet.csv"
    qa_score_sheet_sha = hashlib.sha256(audit_sheet.read_bytes()).hexdigest()
    summary, wrong = score(audit_sheet)
    if hashlib.sha256(audit_sheet.read_bytes()).hexdigest() != qa_score_sheet_sha:
        raise ValueError("Audit sheet changed during scoring")
    target = QA_DESTINATION / "audit30/audit_summary.csv"
    temporary = target.with_suffix(".part")
    with temporary.open("w", encoding="utf8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=["criterion", "checked", "correct", "rate"])
        writer.writeheader()
        writer.writerows(summary)
    os.replace(temporary, target)
    qa_score_generated = True
    print("Review counts:", summary)
    print("Papers with failed criteria:", wrong)
    print("Blank/NA entries are excluded; inspect checked counts before drawing conclusions.")

## Export QA evidence

The original `m2_evidence.zip` is never overwritten.


In [ ]:
if RUN_QA:
    import hashlib
    import json
    import os
    import zipfile

    sheet = QA_DESTINATION / "audit30/audit_sheet.csv"
    score_current = (
        globals().get("qa_score_generated", False)
        and sheet.is_file()
        and hashlib.sha256(sheet.read_bytes()).hexdigest() == globals().get("qa_score_sheet_sha")
    )
    status = {
        "table_check": "passed"
        if globals().get("qa_table_passed", False)
        else "incomplete_or_failed",
        "structure": ("skipped" if not RUN_SOURCE_QA else "complete" if globals().get("qa_structure_complete", False) else "incomplete"),
        "manual_review": ("skipped" if not RUN_SOURCE_QA else "scored_inspect_counts" if score_current else "pending"),
    }
    (QA_DESTINATION / "qa_status.json").write_text(
        json.dumps(status, indent=2) + "\n", encoding="utf8"
    )

    archive = LOCAL / "m2_qa_evidence.zip"
    destination = QA_DESTINATION / "m2_qa_evidence.zip"
    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
        for path in sorted(QA_DESTINATION.rglob("*")):
            if (
                path.is_file()
                and path != destination
                and not path.name.endswith(".part")
                and not any(
                    part.startswith(".qa-sample-")
                    for part in path.relative_to(QA_DESTINATION).parts
                )
            ):
                bundle.write(path, path.relative_to(QA_DESTINATION).as_posix())
    temporary = QA_DESTINATION / ".m2_qa_evidence.part"
    shutil.copy2(archive, temporary)
    os.replace(temporary, destination)
    print("QA evidence:", destination)